# Phase 4: Unsupervised Anomaly Detection
### Using DBSCAN to Cluster 'Agentic' Behavioral Signatures

In this notebook, we move beyond manual rules. We use a density-based clustering algorithm to automatically separate coordinated bot activity from organic human behavior.

In [ ]:
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# 1. Load the data
df = pd.read_csv('../data/raw/transactions.csv')

# 2. Feature Engineering
# Convert timestamp to a numeric value (Unix time) so the model can process it
df['timestamp_numeric'] = pd.to_datetime(df['timestamp']).view('int64') // 10**9
features = df[['timestamp_numeric', 'amount']]

# 3. Scaling
# DBSCAN is distance-based, so we must scale 'Time' and 'Amount' equally
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# 4. Apply DBSCAN
# eps: the 'neighborhood' distance; min_samples: min points to form a cluster
db = DBSCAN(eps=0.1, min_samples=20).fit(scaled_features)
df['cluster'] = db.labels_

# 5. The Reveal
# Cluster -1 is 'Noise' (Human). Clusters 0+ are 'Patterned Behavior' (Bots).
print(f'Detected {len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)} coordinated clusters.')
print('\n--- Fraud Density per Cluster ---')
print(df.groupby('cluster')['is_fraud'].mean())

### Visualization of the Clusters
If our logic is correct, the bots will be tightly packed together, while humans will be 'noise' outliers.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df['timestamp_numeric'], df['amount'], c=df['cluster'], cmap='viridis', s=10, alpha=0.5)
plt.title('DBSCAN Clustering of Transactional Behavior')
plt.xlabel('Time (Numeric)')
plt.ylabel('Amount')
plt.show()